# sweep-hparam-distribution — worked example 1: Log-uniform vs Linear-uniform — choosing the right distribution

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sweep-hparam-distribution`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

For hyperparameters whose effect scales multiplicatively across orders of magnitude — like learning rate or weight decay — `log_uniform_values` samples uniformly on the log scale, giving equal probability to each decade. For bounded probabilities like dropout, a linear `uniform` distribution is more appropriate because the meaningful variation is linear, not logarithmic.

## Worked solution

**Step 1 — Identify the scaling behavior.**
Learning rate matters differently at 1e-4 vs 1e-3 vs 1e-2 — that is, the difference between 0.0001 and 0.001 is as important as the difference between 0.001 and 0.01. This multiplicative structure calls for log-uniform sampling.

**Step 2 — Contrast with linear parameters.**
Dropout rate of 0.1 vs 0.2 is a 0.1-unit additive change, just like 0.3 vs 0.4. There is no multiplicative order-of-magnitude scaling here, so linear `uniform` is correct.

**Step 3 — The spec dicts.**
For log-uniform: `{'distribution': 'log_uniform_values', 'min': low, 'max': high}`. For linear: `{'distribution': 'uniform', 'min': low, 'max': high}`. The `min` and `max` are always in value space, not log space.

**Step 4 — Verify.**
Print each spec and check the distribution name matches the intended scaling. Running a histogram of sampled values would show uniform density on the log scale for the first type and uniform density on the linear scale for the second.

In [ ]:
def lr_spec(low: float = 1e-5, high: float = 1e-1) -> dict:
    """Learning rate: log-uniform (spans orders of magnitude)."""
    return {'distribution': 'log_uniform_values', 'min': low, 'max': high}

def dropout_spec(low: float = 0.0, high: float = 0.5) -> dict:
    """Dropout rate: linear uniform (bounded probability, linear effect)."""
    return {'distribution': 'uniform', 'min': low, 'max': high}

# Demonstrate
print('LR spec:', lr_spec())
print('Dropout spec:', dropout_spec())

# Show what sampling would look like (conceptually)
import math
low, high = 1e-5, 1e-1
log_points = [10 ** (math.log10(low) + i * (math.log10(high) - math.log10(low)) / 5)
              for i in range(6)]
print('\nLog-spaced sample points:', [f'{x:.2e}' for x in log_points])
# Each step is a constant factor, not a constant additive amount